# 🧬 Project 5 — Semantic Correspondence (OFFICIAL FINAL)
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q

# Scaricamento Dataset
!python dataloaders/download_spair.py --root ./data
!python dataloaders/download_pfpascal.py --root ./data

import os, torch, gc
from utils.demo_utils import launch_comparison_demo, launch_robustness_demo
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print('[INFO] GPU Cleared.')

# Auto-detect PF-Pascal root
import shutil as _sh
_macosx = './data/PF-Pascal/__MACOSX'
if os.path.exists(_macosx): _sh.rmtree(_macosx)
_contents = [d for d in os.listdir('./data/PF-Pascal') if os.path.isdir(os.path.join('./data/PF-Pascal', d))]
PASCAL_ROOT = os.path.join('./data/PF-Pascal', _contents[0]) if _contents else './data/PF-Pascal'
print(f'[INFO] PASCAL_ROOT: {PASCAL_ROOT}')

## 🔍 1. Eval Baseline (SPair & PF-Pascal)

In [ ]:
# 1.1 Baselines su SPair-71k
clear_gpu()
print('--- SPair: DINOv2 Baseline ---')
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2.txt
print('--- SPair: DINOv3 Baseline ---')
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov3 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov3.txt
print('--- SPair: SAM Baseline ---')
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone sam_vitb --batch_size 1 --results_file /content/drive/MyDrive/AML/Results/baseline_sam.txt

In [ ]:
# 1.2 Baselines su PF-Pascal (Generalizzazione 0-shot)
clear_gpu()
print('--- PF-Pascal: DINOv2 Baseline ---')
!python evaluate.py --dataset_root "$PASCAL_ROOT" --dataset_type pfpascal --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/gen_pascal_baseline_dinov2.txt
print('--- PF-Pascal: DINOv3 Baseline ---')
!python evaluate.py --dataset_root "$PASCAL_ROOT" --dataset_type pfpascal --baseline_only --backbone dinov3 --results_file /content/drive/MyDrive/AML/Results/gen_pascal_baseline_dinov3.txt

## 🚀 2. Training (LoRA & BitFit)

In [ ]:
# 2.1 Training LoRA (No Curriculum)
if not os.path.exists(f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir "$DRIVE_CKPTS/lora_only"
else: print('[OK] LoRA trovato.')

In [ ]:
# 2.2 Training LoRA (Curriculum Learning)
if not os.path.exists(f'{DRIVE_CKPTS}/lora_curriculum/lora_curriculum_best.pth'):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --use_curriculum --exp_name lora_curriculum --output_dir ./checkpoints/lora_curriculum --backup_dir "$DRIVE_CKPTS/lora_curriculum"
else: print('[OK] LoRA Curriculum trovato.')

In [ ]:
# 2.3 Training BitFit
if not os.path.exists(f'{DRIVE_CKPTS}/bitfit_only/bitfit_only_best.pth'):
    !python train.py --peft_type bitfit --dataset_root ./data/SPair-71k --epochs 5 --exp_name bitfit_only --output_dir ./checkpoints/bitfit_only --backup_dir "$DRIVE_CKPTS/bitfit_only"
else: print('[OK] BitFit trovato.')

## 🎯 3. Raffinamento (Ablation Adaptive Window)

In [ ]:
print('--- LoRA + AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/lora_aw.txt
print('--- LoRA No AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/lora_no_aw.txt
print('--- BitFit + AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/bitfit_aw.txt
print('--- BitFit No AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/bitfit_no_aw.txt

## 🌍 4. Test Finali: PF-Pascal & Temperature Ablation

In [ ]:
# 4.1 Fine-tuned su PF-Pascal
clear_gpu()
print('--- PF-Pascal: LoRA ---')
!python evaluate.py --dataset_root "$PASCAL_ROOT" --dataset_type pfpascal --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/gen_pascal_lora.txt
print('--- PF-Pascal: BitFit ---')
!python evaluate.py --dataset_root "$PASCAL_ROOT" --dataset_type pfpascal --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/gen_pascal_bitfit.txt

In [ ]:
# 4.2 Ablation Temperatura (Calibration vs Regularizzazione)
clear_gpu()
print('--- Ablazione Temperatura su SPair ---')
!python ablate_temperature.py --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/ablation_temperature.txt

In [ ]:
# 4.3 Robustezza Geometrica (15° Rotation su SPair)
clear_gpu()
print('--- LoRA: Rotation 15 deg ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --rotation_deg 15 --results_file /content/drive/MyDrive/AML/Results/lora_rot15.txt
print('--- BitFit: Rotation 15 deg ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --rotation_deg 15 --results_file /content/drive/MyDrive/AML/Results/bitfit_rot15.txt

## 🎨 5. Visual Demos

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

In [ ]:
launch_robustness_demo(ckpt_name='lora_only')